# Train a Text-to-Image Model

This notebook demonstrates how to train a text-to-image model using Hugging Face's Diffusers library.
We'll fine-tune a small Stable Diffusion model on a custom dataset.

## Overview
- **Model**: Stable Diffusion (fine-tuning)
- **Library**: Hugging Face Diffusers + Accelerate
- **Dataset**: Small custom dataset or Hugging Face dataset
- **Task**: Text-to-Image Generation

## 1. Environment Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q diffusers transformers accelerate datasets torch torchvision
!pip install -q xformers  # Optional: for memory-efficient attention
!pip install -q bitsandbytes  # Optional: for 8-bit optimization
!pip install -q wandb  # Optional: for experiment tracking
!pip install -q pillow matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from datasets import load_dataset

import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Training configuration
config = {
    # Model settings
    "model_name": "runwayml/stable-diffusion-v1-5",  # Base model
    "resolution": 512,  # Image resolution
    
    # Training settings
    "batch_size": 1,  # Small batch size for memory efficiency
    "num_epochs": 10,  # Number of training epochs
    "learning_rate": 1e-5,
    "gradient_accumulation_steps": 4,  # Accumulate gradients to simulate larger batch
    "mixed_precision": "fp16",  # Use mixed precision for faster training
    
    # Dataset settings
    "dataset_name": "lambdalabs/pokemon-blip-captions",  # Small dataset example
    "max_train_samples": 100,  # Limit samples for quick training
    
    # Output settings
    "output_dir": "./text-to-image-model",
    "save_steps": 50,
    "logging_steps": 10,
    
    # Seed for reproducibility
    "seed": 42,
}

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)

# Set seed
torch.manual_seed(config["seed"])
np.random.seed(config["seed"])

## 3. Dataset Preparation

In [ ]:
# Load dataset from Hugging Face
dataset = load_dataset(config["dataset_name"], split="train")

# Limit dataset size for quick training
if config["max_train_samples"]:
    dataset = dataset.select(range(min(config["max_train_samples"], len(dataset))))

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")
print(f"\nSample data:")
print(dataset[0])

In [ ]:
# Auto-detect text column name in dataset
def get_text_column(dataset):
    """Detect the text/caption column name in the dataset."""
    possible_names = ['text', 'caption', 'prompt', 'description', 'label']
    for name in possible_names:
        if name in dataset.features:
            return name
    # Fallback: return first string column
    for col, feature in dataset.features.items():
        if hasattr(feature, 'dtype') and feature.dtype == 'string':
            return col
    raise ValueError(f"Could not find text column. Available columns: {list(dataset.features.keys())}")

TEXT_COLUMN = get_text_column(dataset)
print(f"Using text column: '{TEXT_COLUMN}'")

# Visualize some samples from the dataset
def show_dataset_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.ravel()

    for idx in range(min(num_samples, len(dataset))):
        sample = dataset[idx]
        image = sample['image']
        caption = sample[TEXT_COLUMN]

        axes[idx].imshow(image)
        axes[idx].set_title(f"Caption: {caption[:50]}...", fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

show_dataset_samples(dataset)

In [ ]:
# Custom Dataset class
class TextToImageDataset(Dataset):
    def __init__(self, dataset, tokenizer, text_column, resolution=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.resolution = resolution
        self.text_column = text_column
        
        # Image transforms
        self.transforms = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # Normalize to [-1, 1]
        ])
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Get image
        image = item['image']
        if not image.mode == "RGB":
            image = image.convert("RGB")
        image = self.transforms(image)
        
        # Get caption using the detected text column
        caption = item[self.text_column]
        
        # Tokenize caption
        tokens = self.tokenizer(
            caption,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        
        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids[0],
            "caption": caption,
        }

## 4. Model Setup

In [ ]:
# Load pre-trained components
print("Loading model components...")

# Text encoder and tokenizer
tokenizer = CLIPTokenizer.from_pretrained(
    config["model_name"],
    subfolder="tokenizer"
)

text_encoder = CLIPTextModel.from_pretrained(
    config["model_name"],
    subfolder="text_encoder"
)

# VAE (Variational Autoencoder)
vae = AutoencoderKL.from_pretrained(
    config["model_name"],
    subfolder="vae"
)

# UNet (the main model we'll fine-tune)
unet = UNet2DConditionModel.from_pretrained(
    config["model_name"],
    subfolder="unet"
)

# Noise scheduler
noise_scheduler = DDPMScheduler.from_pretrained(
    config["model_name"],
    subfolder="scheduler"
)

print("Model components loaded successfully!")

In [ ]:
# Freeze VAE and text encoder (we only train the UNet)
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

# Enable gradient checkpointing for memory efficiency
unet.enable_gradient_checkpointing()

print(f"Trainable parameters in UNet: {sum(p.numel() for p in unet.parameters() if p.requires_grad):,}")

## 5. Training Setup

In [ ]:
# Initialize accelerator for distributed training and mixed precision
accelerator = Accelerator(
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    mixed_precision=config["mixed_precision"],
)

# Create dataset and dataloader
train_dataset = TextToImageDataset(
    dataset=dataset,
    tokenizer=tokenizer,
    text_column=TEXT_COLUMN,  # Pass the detected text column
    resolution=config["resolution"]
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,
)

# Optimizer
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=config["learning_rate"],
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    eps=1e-08,
)

# Prepare everything with accelerator
unet, optimizer, train_dataloader = accelerator.prepare(
    unet, optimizer, train_dataloader
)

# Move models to device
vae.to(accelerator.device)
text_encoder.to(accelerator.device)

print(f"Training on device: {accelerator.device}")
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_dataloader)}")

## 6. Training Loop

In [ ]:
# Training function
def train_epoch(epoch):
    unet.train()
    total_loss = 0
    
    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/{config['num_epochs']}",
        disable=not accelerator.is_local_main_process
    )
    
    for step, batch in enumerate(progress_bar):
        with accelerator.accumulate(unet):
            # Convert images to latent space
            latents = vae.encode(batch["pixel_values"]).latent_dist.sample()
            latents = latents * 0.18215  # Scaling factor
            
            # Sample noise
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            
            # Sample random timesteps
            timesteps = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (bsz,),
                device=latents.device
            )
            timesteps = timesteps.long()
            
            # Add noise to latents
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            
            # Get text embeddings
            encoder_hidden_states = text_encoder(batch["input_ids"])[0]
            
            # Predict noise
            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            
            # Calculate loss
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
            
            # Backpropagation
            accelerator.backward(loss)
            
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(unet.parameters(), 1.0)
            
            optimizer.step()
            optimizer.zero_grad()
        
        # Update progress bar
        total_loss += loss.detach().item()
        avg_loss = total_loss / (step + 1)
        progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
        
        # Logging
        if step % config["logging_steps"] == 0:
            if accelerator.is_local_main_process:
                print(f"Step {step}: Loss = {loss.item():.4f}")
    
    return total_loss / len(train_dataloader)

In [ ]:
# Train the model
print("Starting training...\n")

train_losses = []

for epoch in range(config["num_epochs"]):
    epoch_loss = train_epoch(epoch)
    train_losses.append(epoch_loss)
    
    print(f"\nEpoch {epoch + 1}/{config['num_epochs']} - Average Loss: {epoch_loss:.4f}\n")
    
    # Save checkpoint
    if (epoch + 1) % 5 == 0 or epoch == config["num_epochs"] - 1:
        if accelerator.is_local_main_process:
            checkpoint_dir = os.path.join(config["output_dir"], f"checkpoint-epoch-{epoch + 1}")
            os.makedirs(checkpoint_dir, exist_ok=True)
            
            # Unwrap model from accelerator
            unwrapped_unet = accelerator.unwrap_model(unet)
            unwrapped_unet.save_pretrained(checkpoint_dir)
            
            print(f"Checkpoint saved to {checkpoint_dir}")

print("\nTraining completed!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.grid(True)
plt.savefig(os.path.join(config["output_dir"], 'training_loss.png'))
plt.show()

## 7. Save Final Model

In [ ]:
# Save the final model
if accelerator.is_local_main_process:
    final_model_dir = os.path.join(config["output_dir"], "final_model")
    os.makedirs(final_model_dir, exist_ok=True)
    
    # Save UNet
    unwrapped_unet = accelerator.unwrap_model(unet)
    unwrapped_unet.save_pretrained(final_model_dir)
    
    # Save full pipeline for easy inference
    pipeline = StableDiffusionPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unwrapped_unet,
        scheduler=noise_scheduler,
        safety_checker=None,
        feature_extractor=None,
    )
    pipeline.save_pretrained(os.path.join(config["output_dir"], "pipeline"))
    
    print(f"Final model saved to {config['output_dir']}")

## 8. Inference and Image Generation

In [ ]:
# Load the trained pipeline
pipeline = StableDiffusionPipeline.from_pretrained(
    os.path.join(config["output_dir"], "pipeline"),
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
pipeline = pipeline.to(accelerator.device)

print("Pipeline loaded successfully!")

In [ ]:
# Generate images from text prompts
def generate_images(prompts, num_inference_steps=50, guidance_scale=7.5):
    """
    Generate images from text prompts using the trained model.
    
    Args:
        prompts: List of text prompts
        num_inference_steps: Number of denoising steps
        guidance_scale: Guidance scale for classifier-free guidance
    """
    images = []
    
    with torch.no_grad():
        for prompt in prompts:
            image = pipeline(
                prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
            ).images[0]
            images.append(image)
    
    return images

In [ ]:
# Test prompts (customize based on your training data)
test_prompts = [
    "a cute pokemon with big eyes",
    "a fire type pokemon",
    "a water type pokemon swimming",
    "a grass type pokemon in a forest",
]

print("Generating images...")
generated_images = generate_images(test_prompts)

# Display generated images
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.ravel()

for idx, (image, prompt) in enumerate(zip(generated_images, test_prompts)):
    axes[idx].imshow(image)
    axes[idx].set_title(f"Prompt: {prompt}", fontsize=10)
    axes[idx].axis('off')
    
    # Save image
    image.save(os.path.join(config["output_dir"], f"generated_{idx}.png"))

plt.tight_layout()
plt.savefig(os.path.join(config["output_dir"], 'generated_images.png'))
plt.show()

print(f"\nImages saved to {config['output_dir']}")

## 9. Model Evaluation

In [ ]:
# Compare with base model (optional)
base_pipeline = StableDiffusionPipeline.from_pretrained(
    config["model_name"],
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
base_pipeline = base_pipeline.to(accelerator.device)

# Generate comparison images
comparison_prompt = "a cute pokemon with big eyes"

print(f"Comparing fine-tuned model vs base model for prompt: '{comparison_prompt}'")

with torch.no_grad():
    finetuned_image = pipeline(comparison_prompt, num_inference_steps=50).images[0]
    base_image = base_pipeline(comparison_prompt, num_inference_steps=50).images[0]

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(base_image)
axes[0].set_title("Base Model", fontsize=12)
axes[0].axis('off')

axes[1].imshow(finetuned_image)
axes[1].set_title("Fine-tuned Model", fontsize=12)
axes[1].axis('off')

plt.suptitle(f"Prompt: {comparison_prompt}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config["output_dir"], 'model_comparison.png'))
plt.show()

## 10. Interactive Generation

In [ ]:
# Interactive image generation
def interactive_generate():
    """
    Interactive function to generate images from custom prompts.
    """
    print("Enter your prompt (or 'quit' to exit):")
    
    while True:
        prompt = input("Prompt: ")
        
        if prompt.lower() == 'quit':
            break
        
        print("Generating image...")
        with torch.no_grad():
            image = pipeline(prompt, num_inference_steps=50).images[0]
        
        # Display
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        plt.title(f"Prompt: {prompt}")
        plt.axis('off')
        plt.show()

# Uncomment to run interactive mode
# interactive_generate()

## Summary

This notebook demonstrated:
1. Setting up the environment for text-to-image training
2. Loading and preparing a dataset
3. Fine-tuning a Stable Diffusion model
4. Generating images from text prompts
5. Evaluating and comparing the model

### Next Steps:
- Experiment with larger datasets
- Adjust hyperparameters (learning rate, batch size, epochs)
- Try different base models
- Implement LoRA for more efficient fine-tuning
- Add more sophisticated evaluation metrics
- Deploy the model for production use

In [ ]:
# Clean up (optional)
# import gc
# del pipeline, base_pipeline, unet, vae, text_encoder
# gc.collect()
# torch.cuda.empty_cache()